# Camada Silver - Marketing Funnel by Olist + IPCA (Bruno)

Limpa as tabelas Bronze e grava as tabelas Silver em Delta, na base `silver` do grupo.

| Tabela Silver | 1 linha = | Origem (Bronze) | Usada no modelo em |
|---|---|---|---|
| `silver.funil_leads` | 1 lead (MQL) com primeiro contacto entre junho e dezembro de 2017, com os dados do negócio fechado se converteu | `funil_mql` + `funil_closed_deals` | `fact_funil_marketing`, `dim_perfil_lead`, `dim_vendedor` |
| `silver.ipca_mensal` | 1 mês (IPCA a 12 meses do Brasil) | `ipca` | `dim_ipca_mensal` |

Regras do grupo aplicadas:
- **Intervalo comum:** só entram leads com primeiro contacto de 2017-06-01 a 2017-12-31, e os meses do IPCA do mesmo intervalo.
- **Valores em falta ficam NULL na Silver**; na Gold passam ao registo `-1` da dimensão (deixa de haver o texto "unknown").
- A data de fecho (`won_date`) é mantida mesmo quando cai depois de dezembro de 2017: no modelo é um atributo (`data_fecho`), não a data do facto.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Silver - Marketing Funnel + IPCA (Bruno)") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [2]:
# Intervalo de tempo comum do grupo (README): só entram registos de 1 de junho a 31 de dezembro de 2017
DATA_INICIO = "2017-06-01"
DATA_FIM = "2017-12-31"

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver LOCATION 'hdfs://hdfs-nn:9000/warehouse/silver.db/'")

def gravar_silver(df, tabela):
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/silver.db/{tabela}/") \
        .saveAsTable(f"silver.{tabela}")
    print(f"silver.{tabela}: {spark.table(f'silver.{tabela}').count()} linhas")

def nulos(df):
    """Número de nulos por coluna."""
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

def texto(c):
    """Remove espaços nas pontas e passa texto vazio a NULL."""
    t = F.trim(F.col(c))
    return F.when(t == "", None).otherwise(t)

## Leads (MQL) e negócios fechados

In [4]:
mql = spark.table("bronze.funil_mql")
deals = spark.table("bronze.funil_closed_deals")
print("MQL na Bronze:", mql.count(), "| negócios fechados na Bronze:", deals.count())

# Qualidade na origem: chaves duplicadas e negócios sem lead
print("mql_id duplicados (MQL):", mql.count() - mql.select("mql_id").distinct().count())
print("mql_id duplicados (negócios):", deals.count() - deals.select("mql_id").distinct().count())
print("negócios sem lead correspondente:", deals.join(mql, "mql_id", "left_anti").count())
mql.select(F.min("first_contact_date"), F.max("first_contact_date")).show()
deals.select(F.min("won_date"), F.max("won_date")).show()

MQL na Bronze: 8000 | negócios fechados na Bronze: 842
mql_id duplicados (MQL): 0
mql_id duplicados (negócios): 0
negócios sem lead correspondente: 0
+-----------------------+-----------------------+
|min(first_contact_date)|max(first_contact_date)|
+-----------------------+-----------------------+
|             2017-06-14|             2018-05-31|
+-----------------------+-----------------------+

+-------------------+-------------------+
|      min(won_date)|      max(won_date)|
+-------------------+-------------------+
|2017-12-05 02:00:00|2018-11-14 18:04:19|
+-------------------+-------------------+



In [5]:
# Valores originais dos campos categóricos (para decidir a limpeza)
for c in ["origin"]:
    mql.groupBy(c).count().orderBy(F.desc("count")).show(20, truncate=False)
for c in ["business_segment", "lead_type", "lead_behaviour_profile", "has_company", "has_gtin", "average_stock", "business_type"]:
    deals.groupBy(c).count().orderBy(F.desc("count")).show(40, truncate=False)

+-----------------+-----+
|origin           |count|
+-----------------+-----+
|organic_search   |2296 |
|paid_search      |1586 |
|social           |1350 |
|unknown          |1099 |
|direct_traffic   |499  |
|email            |493  |
|referral         |284  |
|other            |150  |
|display          |118  |
|other_publicities|65   |
|null             |60   |
+-----------------+-----+

+-------------------------------+-----+
|business_segment               |count|
+-------------------------------+-----+
|home_decor                     |105  |
|health_beauty                  |93   |
|car_accessories                |77   |
|household_utilities            |71   |
|construction_tools_house_garden|69   |
|audio_video_electronics        |64   |
|computers                      |34   |
|pet                            |30   |
|food_supplement                |28   |
|food_drink                     |26   |
|sports_leisure                 |25   |
|bed_bath_table                 |22   |
|bags_bac

In [6]:
def booleano(c):
    """has_company / has_gtin vêm como texto ('True'/'False'); passam a BOOLEAN."""
    t = F.lower(F.trim(F.col(c)))
    return F.when(t == "true", True).when(t == "false", False).otherwise(F.lit(None).cast("boolean"))

def categoria(c):
    """Minúsculas, sem espaços nas pontas; vazio e 'unknown' passam a NULL."""
    t = F.lower(F.trim(F.col(c)))
    return F.when((t == "") | (t == "unknown"), None).otherwise(t)

mql_limpo = (mql
    .select(
        texto("mql_id").alias("mql_id"),
        F.to_date("first_contact_date", "yyyy-MM-dd").alias("data_primeiro_contacto"),
        texto("landing_page_id").alias("landing_page_id"),
        categoria("origin").alias("origem"))
    .filter(F.col("data_primeiro_contacto").between(DATA_INICIO, DATA_FIM)))

deals_limpo = deals.select(
    texto("mql_id").alias("mql_id"),
    texto("seller_id").alias("seller_id"),
    texto("sdr_id").alias("sdr_id"),
    texto("sr_id").alias("sr_id"),
    F.to_date(F.to_timestamp("won_date", "yyyy-MM-dd HH:mm:ss")).alias("data_fecho"),
    categoria("business_segment").alias("segmento_negocio"),
    categoria("lead_type").alias("tipo_lead"),
    categoria("lead_behaviour_profile").alias("perfil_comportamento"),
    booleano("has_company").alias("tem_empresa"),
    booleano("has_gtin").alias("tem_gtin"),
    categoria("average_stock").alias("stock_medio"),
    categoria("business_type").alias("tipo_negocio"),
    F.col("declared_product_catalog_size").cast("double").cast("int").alias("tamanho_catalogo"),
    F.col("declared_monthly_revenue").cast("decimal(14,2)").alias("receita_mensal_declarada"))

leads = (mql_limpo
    .join(deals_limpo, "mql_id", "left")
    .withColumn("is_convertido", F.col("seller_id").isNotNull())
    .withColumn("dias_ate_fecho", F.datediff("data_fecho", "data_primeiro_contacto")))

print("Leads com primeiro contacto em jun-dez 2017:", leads.count(), "de", mql.count())
print("Convertidas:", leads.filter("is_convertido").count())
leads.groupBy(F.date_format("data_primeiro_contacto", "yyyy-MM").alias("mes")).count().orderBy("mes").show()

Leads com primeiro contacto em jun-dez 2017: 2002 de 8000
Convertidas: 61
+-------+-----+
|    mes|count|
+-------+-----+
|2017-06|    4|
|2017-07|  239|
|2017-08|  386|
|2017-09|  312|
|2017-10|  416|
|2017-11|  445|
|2017-12|  200|
+-------+-----+



In [7]:
# Validações antes de gravar
assert leads.count() == leads.select("mql_id").distinct().count(), "mql_id repetido"
assert leads.filter(~F.col("data_primeiro_contacto").between(DATA_INICIO, DATA_FIM)).count() == 0
assert leads.filter(F.col("dias_ate_fecho") < 0).count() == 0, "fecho antes do primeiro contacto"
nulos(leads)
gravar_silver(leads, "funil_leads")

-RECORD 0------------------------
 mql_id                   | 0    
 data_primeiro_contacto   | 0    
 landing_page_id          | 0    
 origem                   | 298  
 seller_id                | 1941 
 sdr_id                   | 1941 
 sr_id                    | 1941 
 data_fecho               | 1941 
 segmento_negocio         | 1942 
 tipo_lead                | 1942 
 perfil_comportamento     | 1964 
 tem_empresa              | 1984 
 tem_gtin                 | 1984 
 stock_medio              | 1987 
 tipo_negocio             | 1941 
 tamanho_catalogo         | 1986 
 receita_mensal_declarada | 1941 
 is_convertido            | 0    
 dias_ate_fecho           | 1941 

silver.funil_leads: 2002 linhas


## IPCA mensal

O ficheiro tem a data no formato `MM/AAAA` e a taxa a 12 meses em percentagem. A chave do mês fica no formato
`AAAAMM` (`ano_mes`), igual ao `sk_mes` da `dim_ipca_mensal` do modelo.

In [8]:
ipca = spark.table("bronze.ipca")
ipca_limpo = (ipca
    .select(
        F.to_date(F.concat(F.lit("01/"), F.trim(F.col("Date"))), "dd/MM/yyyy").alias("mes_referencia"),
        F.col("IPCA Rate (last 12 months)").cast("decimal(8,2)").alias("ipca_12m"))
    .withColumn("ano_mes", (F.year("mes_referencia") * 100 + F.month("mes_referencia")).cast("int"))
    .filter(F.col("mes_referencia").between(DATA_INICIO, DATA_FIM))
    .select("ano_mes", "mes_referencia", "ipca_12m")
    .orderBy("ano_mes"))

assert ipca_limpo.count() == 7, "Esperavam-se os 7 meses de junho a dezembro de 2017"
assert ipca_limpo.filter(F.col("ipca_12m").isNull()).count() == 0
ipca_limpo.show()
gravar_silver(ipca_limpo, "ipca_mensal")

+-------+--------------+--------+
|ano_mes|mes_referencia|ipca_12m|
+-------+--------------+--------+
| 201706|    2017-06-01|    3.00|
| 201707|    2017-07-01|    2.71|
| 201708|    2017-08-01|    2.46|
| 201709|    2017-09-01|    2.54|
| 201710|    2017-10-01|    2.70|
| 201711|    2017-11-01|    2.80|
| 201712|    2017-12-01|    2.95|
+-------+--------------+--------+

silver.ipca_mensal: 7 linhas


## Resultado

In [9]:
spark.sql("""
    SELECT COUNT(*) AS leads,
           SUM(CAST(is_convertido AS INT)) AS convertidas,
           ROUND(AVG(CAST(is_convertido AS INT)) * 100, 2) AS taxa_conversao_pct,
           ROUND(AVG(dias_ate_fecho), 1) AS media_dias_ate_fecho
    FROM silver.funil_leads
""").show()
spark.sql("""
    SELECT origem, COUNT(*) AS leads, ROUND(AVG(CAST(is_convertido AS INT)) * 100, 1) AS taxa_conversao_pct
    FROM silver.funil_leads GROUP BY origem ORDER BY leads DESC
""").show(truncate=False)

+-----+-----------+------------------+--------------------+
|leads|convertidas|taxa_conversao_pct|media_dias_ate_fecho|
+-----+-----------+------------------+--------------------+
| 2002|         61|              3.05|               212.2|
+-----+-----------+------------------+--------------------+

+-----------------+-----+------------------+
|origem           |leads|taxa_conversao_pct|
+-----------------+-----+------------------+
|organic_search   |560  |2.9               |
|paid_search      |404  |5.0               |
|social           |299  |3.0               |
|null             |298  |4.0               |
|email            |146  |0.7               |
|direct_traffic   |119  |1.7               |
|referral         |73   |0.0               |
|display          |43   |2.3               |
|other            |36   |0.0               |
|other_publicities|24   |0.0               |
+-----------------+-----+------------------+



In [10]:
spark.stop()